# Angular-mode onset

Companion notebook for Fig. 3c–e. Explore the cached structures or scores used in these panels; this notebook does not reproduce the final figure layout.


## What this demonstrates

Four displacement fields are matched in coordinate-space amplitude but differ in angular order. For these four prescribed fields, the scalar C6 self-match residual alone does not uniquely identify the mode. The cached degree-resolved response exhibits an onset consistent with the angular-order prediction for this controlled construction; this is not a universal guarantee for arbitrary finite deformations or encoders.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "demo" / "data" / "manifest.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the SymFP repository or its demo directory.")
CODE_ROOT = PROJECT_ROOT / "code" if (PROJECT_ROOT / "code" / "utils" / "demo_utils.py").is_file() else PROJECT_ROOT
sys.path.insert(0, str(CODE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

from utils.demo_utils import (
    ANGULAR_MODES,
    DEGREE_STYLES,
    PALETTE,
    RANK_ACTIVATION_CASES,
    SUBGROUP_BOUNDARIES,
    SUBGROUP_COLUMNS,
    SUBGROUP_ENTERED,
    SUBGROUP_EXPECTED_ONSETS,
    SUBGROUP_TARGETS,
    environment_versions,
    load_angular_mode,
    load_rank_activation,
    load_subgroup_chain,
    runtime_summary,
)

runtime = runtime_summary()
display(environment_versions().style.hide(axis="index"))
print(f"Project root: {runtime['project_root']}")
print(f"Device available to the project: {runtime['device_available']}")

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#555555",
    "axes.linewidth": 0.8,
    "figure.dpi": 120,
})

## Configuration

Choose one of the four published displacement fields and re-run the notebook.

In [ ]:
MODE = "IP1_ellipse"
assert MODE in ANGULAR_MODES
config = ANGULAR_MODES[MODE]
print(f"Selected mode: {config['label']} (C6 → {config['child']})")
print(f"Angular-order lower bound: l ≥ {config['angular_order']}")

## Load data

The notebook reads 21 amplitudes from 0 to 0.3, the cached C6 self-match residual and the degree-resolved relative tail energies used for Fig. 3c–e.

In [ ]:
mode_frames, verdict, positions, subunit_index = load_angular_mode(MODE)
ONSET_THRESHOLD = float(verdict["onset_tolerance"])
last = mode_frames.iloc[-1]
active_degrees = [l for l in range(1, 7) if last[f"ratio_{l}{'o' if l % 2 else 'e'}"] > ONSET_THRESHOLD]
observed_onset = min(active_degrees) if active_degrees else -1
assert observed_onset == int(verdict["onset_degree_per_mode"][MODE]), "Cached verdict disagrees with the result table"
assert len(positions) == len(mode_frames) == 21
print(f"Loaded {len(mode_frames)} amplitudes: {mode_frames['alpha'].min():.3f}–{mode_frames['alpha'].max():.3f}.")

## Plot

The structure pair shows the parent and maximum-amplitude field. The scalar panel shows the C6 self-match RMSD. The spectral panel shows the relative tail energy at each degree; the grey line marks the operational onset threshold (1e-6); values below 1e-12 are clipped for display.

In [ ]:
def infer_edges(coords: np.ndarray, scale: float = 1.18):
    """Infer only the nearest-neighbour skeleton for visualisation."""
    diff = coords[:, None, :] - coords[None, :, :]
    distances = np.linalg.norm(diff, axis=-1)
    distances[distances == 0] = np.inf
    cutoff = scale * np.median(distances.min(axis=1))
    return [(i, j) for i in range(len(coords)) for j in range(i + 1, len(coords))
            if distances[i, j] <= cutoff]


def set_equal_3d(ax, coords: np.ndarray):
    centre = coords.mean(axis=0)
    radius = np.ptp(coords, axis=0).max() / 2
    radius = max(radius, 1e-6)
    ax.set_xlim(centre[0] - radius, centre[0] + radius)
    ax.set_ylim(centre[1] - radius, centre[1] + radius)
    ax.set_zlim(centre[2] - radius, centre[2] + radius)
    ax.set_box_aspect((1, 1, 1))


def draw_structure(ax, coords: np.ndarray, title: str):
    for i, j in infer_edges(coords):
        ax.plot(*zip(coords[i], coords[j]), color=PALETTE["grey"], lw=1.2, zorder=1)
    ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], s=32, facecolor="white",
               edgecolor=PALETTE["ink"], linewidth=1.0, depthshade=False, zorder=2)
    set_equal_3d(ax, coords)
    ax.view_init(elev=24, azim=-58)
    ax.set_title(title, pad=6)
    ax.set_axis_off()

In [ ]:
def draw_mode_structure(ax, coords: np.ndarray, title: str):
    centroids = np.stack([coords[subunit_index == k].mean(axis=0)
                          for k in range(int(subunit_index.max()) + 1)])
    order = np.argsort(np.arctan2(centroids[:, 1], centroids[:, 0]))
    ring = centroids[order]
    ring = np.vstack([ring, ring[0]])
    ax.plot(ring[:, 0], ring[:, 1], ring[:, 2], color=PALETTE["grey"], lw=1.5)
    ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], s=20, facecolor="white",
               edgecolor=PALETTE["ink"], linewidth=0.9, depthshade=False)
    ax.scatter(centroids[:, 0], centroids[:, 1], centroids[:, 2], s=38, facecolor="white",
               edgecolor=PALETTE["ink"], linewidth=1.1, depthshade=False)
    set_equal_3d(ax, coords)
    ax.view_init(elev=24, azim=-58)
    ax.set_title(title, pad=6)
    ax.set_axis_off()


fig = plt.figure(figsize=(11, 6.2), constrained_layout=True)
gs = fig.add_gridspec(2, 3, height_ratios=[1, 1.1])

draw_mode_structure(fig.add_subplot(gs[0, 0], projection="3d"), positions[0], "C6 parent")
draw_mode_structure(fig.add_subplot(gs[0, 1], projection="3d"), positions[-1],
                    f"{config['label']} → {config['child']}")

ax_scalar = fig.add_subplot(gs[0, 2])
ax_scalar.plot(mode_frames["alpha"], mode_frames["c6_selfmatch_rmsd"],
               color=PALETTE["coral"] if config["child"] in {"C1", "C2"} else PALETTE["blue"],
               lw=2.2)
ax_scalar.set_xlabel("amplitude")
ax_scalar.set_ylabel("C6 self-match RMSD")
ax_scalar.set_title("Single-operation residual")

ax_tail = fig.add_subplot(gs[1, :])
for degree in range(1, 7):
    column = f"ratio_{degree}{'o' if degree % 2 else 'e'}"
    colour, linestyle = DEGREE_STYLES[degree]
    emphasis = degree == observed_onset
    ax_tail.plot(mode_frames["alpha"], np.maximum(mode_frames[column], 1e-12),
                 color=colour, ls=linestyle, lw=2.7 if emphasis else 1.6,
                 label=f"l={degree}")
ax_tail.axhline(ONSET_THRESHOLD, color=PALETTE["grey"], ls="--", lw=0.9)
ax_tail.set_yscale("log")
ax_tail.set_xlabel("amplitude")
ax_tail.set_ylabel("relative tail energy")
ax_tail.legend(frameon=False, ncol=6)
plt.show()

In [ ]:
observed = observed_onset
summary = pd.DataFrame([{
    "mode": config["label"],
    "child group": config["child"],
    "angular-order lower bound": config["angular_order"],
    "observed onset degree": observed,
    "attains the bound": observed == config["angular_order"],
    "C6 self-match RMSD at max amplitude": float(last["c6_selfmatch_rmsd"]),
}])
display(summary.style.hide(axis="index"))

## Short conclusion

In [ ]:
observed = observed_onset
if observed >= config["angular_order"]:
    relation = "attains" if observed == config["angular_order"] else "lies above"
    print(f"PASS — the observed onset l={observed} {relation} the angular-order prediction for this prescribed field l≥{config['angular_order']}.")
else:
    print("CHECK — the observed onset lies below the angular-order lower bound.")
print("This check uses cached spectral results for the prescribed fields; it does not identify an arbitrary displacement field.")